# Futbolcu Veri Madenciliği ve Makine Öğrenmesi

Bu notebook, Veri Madenciliği dersi kapsamında hazırlanmış orijinal çalışmanın **yeniden düzenlenmiş portföy sürümüdür**.

Ham veri seti bu repo içerisinde arşivlenmemiştir. Kod akışı ve model sonuçları daha önce alınmış proje çıktılarından korunmuştur.

> Aşağıda belirtilen sayısal sonuçlar yeniden çalıştırılmış sonuçlar değil, orijinal proje çıktısında yer alan sonuçlardır.


## 1. Kütüphaneler ve veri setinin yüklenmesi

Orijinal projede `archive.zip` içinde üç CSV dosyası kullanılmıştır:

- `player_predictions.csv`
- `goalkeeper_predictions.csv`
- `player_role_clusters.csv`

Orijinal boyutlar:
- Futbolcular: **(30645, 58)**
- Kaleciler: **(2567, 58)**
- Oyuncu rol kümeleri: **(11273, 18)**


In [ ]:
from pathlib import Path
import os
import re
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

ARCHIVE_PATH = Path("../data/archive.zip")
EXTRACT_DIR = Path("../data/extracted")

if not ARCHIVE_PATH.exists():
    raise FileNotFoundError(
        "Raw dataset archive is not available. "
        "Restore it as data/archive.zip before running the notebook."
    )

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ARCHIVE_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

df_player = pd.read_csv(EXTRACT_DIR / "player_predictions.csv", encoding="utf-8")
df_gk = pd.read_csv(EXTRACT_DIR / "goalkeeper_predictions.csv", encoding="utf-8")
df_cluster = pd.read_csv(EXTRACT_DIR / "player_role_clusters.csv", encoding="utf-8")

print("player:", df_player.shape)
print("goalkeeper:", df_gk.shape)
print("clusters:", df_cluster.shape)

df_player.head()


## 2. Eksik veri analizi

Orijinal çıktıda:

- Futbolcularda `goalsave_score` sütunu tamamen eksikti.
- Kalecilerde `goals_score`, `assists_score` ve `shots_score` sütunları tamamen eksikti.

Bu alanlar role özel olduğundan, birleştirme öncesinde rol bazlı şekilde ele alınmıştır.


In [ ]:
print("PLAYER missing values?:", df_player.isna().any().any())
print("GK missing values?:", df_gk.isna().any().any())

missing_player = df_player.isna().sum().sort_values(ascending=False)
missing_gk = df_gk.isna().sum().sort_values(ascending=False)

print("\nPLAYER - top missing columns:")
print(missing_player.head(15))

print("\nGK - top missing columns:")
print(missing_gk.head(15))


## 3. Rol bazlı veri temizleme ve birleştirme

`is_goalkeeper` isminde yeni bir özellik oluşturulmuştur:

- `0` → saha oyuncusu
- `1` → kaleci

Role ait olmayan skor alanları `0` ile doldurulduktan sonra veri setleri birleştirilmiştir.

**Orijinal birleşmiş veri boyutu:** `(33212, 59)`


In [ ]:
df_player_fix = df_player.copy()
df_gk_fix = df_gk.copy()

df_player_fix["is_goalkeeper"] = 0
df_gk_fix["is_goalkeeper"] = 1

if "goalsave_score" in df_player_fix.columns:
    df_player_fix["goalsave_score"] = df_player_fix["goalsave_score"].fillna(0)

for c in ["goals_score", "assists_score", "shots_score"]:
    if c in df_gk_fix.columns:
        df_gk_fix[c] = df_gk_fix[c].fillna(0)

df_all = pd.concat(
    [df_player_fix, df_gk_fix],
    ignore_index=True,
    sort=False
)

print("Merged data:", df_all.shape)
print("Total NaN:", int(df_all.isna().sum().sum()))


## 4. Boy ve kilo verilerinin dönüştürülmesi

Kaynak veri setindeki `"175 cm"` ve `"72 kg"` gibi metin değerleri sayısal değerlere dönüştürülmüştür.

Yeni sütunlar:

- `height_cm`
- `weight_kg`

Gerçek dışı `0` değerleri `NaN` olarak işaretlenmiş, ardından sayısal eksik değerler median ile doldurulmuştur.


In [ ]:
def extract_number(x):
    if pd.isna(x):
        return np.nan
    m = re.search(r"[-+]?\d*\.?\d+", str(x).replace(",", "."))
    return float(m.group()) if m else np.nan

df_all["height_cm"] = df_all["height"].apply(extract_number)
df_all["weight_kg"] = df_all["weight"].apply(extract_number)

df_all.drop(columns=["height", "weight"], inplace=True)

df_all.loc[df_all["height_cm"] <= 0, "height_cm"] = np.nan
df_all.loc[df_all["weight_kg"] <= 0, "weight_kg"] = np.nan

num_cols = df_all.select_dtypes(include=["number"]).columns
df_all[num_cols] = df_all[num_cols].fillna(df_all[num_cols].median())

df_all[["height_cm", "weight_kg"]].describe()


## 5. Normalizasyon ve standardizasyon

Orijinal çalışmada iki farklı ölçekleme yöntemi uygulanmıştır:

- **Min-Max Scaling**
- **Z-score Standardization**

`id`, `season` ve `age` gibi alanlar ölçeklendirme dışında bırakılmıştır.


In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

num_cols = df_all.select_dtypes(include=["number"]).columns.tolist()

exclude = {"id", "season", "age"}
scale_cols = [c for c in num_cols if c not in exclude]

mm = MinMaxScaler()
df_all_mm = df_all.copy()
df_all_mm[scale_cols] = mm.fit_transform(df_all_mm[scale_cols])

zs = StandardScaler()
df_all_z = df_all.copy()
df_all_z[scale_cols] = zs.fit_transform(df_all_z[scale_cols])

print("Scaled column count:", len(scale_cols))


## 6. Veri azaltma

Modelleme açısından gereksiz olan `id` ve `name` sütunları çıkarılmıştır.

**Orijinal sonuç:** `(33212, 57)`


In [ ]:
drop_cols = [c for c in ["id", "name"] if c in df_all_mm.columns]
df_reduced = df_all_mm.drop(columns=drop_cols)

print("Before:", df_all_mm.shape)
print("After:", df_reduced.shape)


## 7. IQR yöntemi ile aykırı değer analizi

Aykırı değer sınırları klasik IQR yöntemi ile hesaplanmıştır:

- Alt sınır = `Q1 - 1.5 × IQR`
- Üst sınır = `Q3 + 1.5 × IQR`

Sınırların dışında kalan değerler tespit edilmiş ve clipping uygulanmıştır.


In [ ]:
def iqr_outlier_report(df, cols):
    report = []
    for c in cols:
        s = df[c].dropna()
        if s.empty:
            continue

        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1

        if iqr == 0:
            low, high = q1, q3
            count = 0
        else:
            low = q1 - 1.5 * iqr
            high = q3 + 1.5 * iqr
            count = int(((s < low) | (s > high)).sum())

        report.append((c, count, low, high))

    return sorted(report, key=lambda x: x[1], reverse=True)

base_df = df_reduced.copy()
num_cols = base_df.select_dtypes(include=["number"]).columns.tolist()

df_iqr_clean = base_df.copy()
iqr_rows = []

for c in num_cols:
    q1 = df_iqr_clean[c].quantile(0.25)
    q3 = df_iqr_clean[c].quantile(0.75)
    iqr = q3 - q1

    if pd.isna(iqr) or iqr == 0:
        continue

    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr

    outlier_count = ((df_iqr_clean[c] < low) | (df_iqr_clean[c] > high)).sum()
    df_iqr_clean[c] = df_iqr_clean[c].clip(lower=low, upper=high)

    iqr_rows.append([c, int(outlier_count), float(low), float(high)])

iqr_report = (
    pd.DataFrame(iqr_rows, columns=["column", "outlier_count", "low", "high"])
    .sort_values("outlier_count", ascending=False)
)

display(iqr_report.head(15))


## 8. Rolling Median ile yumuşatma

`minutes_played` ve `rating_pred` sütunlarında pencere boyutu `5` olan rolling median uygulanmıştır.

Orijinal çıktıda `minutes_played` standart sapması yaklaşık **0.2434** değerinden **0.1771** değerine düşmüştür.


In [ ]:
df_smooth = df_reduced.copy()

smooth_cols = [
    c for c in ["minutes_played", "rating_pred"]
    if c in df_smooth.columns
]

for c in smooth_cols:
    df_smooth[c + "_smooth"] = (
        df_smooth[c]
        .rolling(window=5, min_periods=1)
        .median()
    )

print("Smoothed columns:", [c + "_smooth" for c in smooth_cols])


## 9. Bayesian Network için ayrıklaştırma

Bazı sayısal değişkenler kategorik sınıflara dönüştürülmüştür.

`rating_pred` dört quantile sınıfına ayrılmıştır:

- `low`
- `mid`
- `high`
- `elite`


In [ ]:
df_bn = df_iqr_clean.copy()

if "age" in df_bn.columns:
    df_bn["age_bin"] = pd.cut(
        df_bn["age"],
        bins=[-1, 20, 25, 30, 35, 40, 200],
        labels=["<=20", "21-25", "26-30", "31-35", "36-40", "41+"]
    ).astype(str)

if "minutes_played" in df_bn.columns:
    df_bn["minutes_played_bin"] = pd.qcut(
        df_bn["minutes_played"],
        q=4,
        labels=["low", "mid", "high", "elite"]
    ).astype(str)

if "rating_pred" in df_bn.columns:
    df_bn["rating_pred_bin"] = pd.qcut(
        df_bn["rating_pred"],
        q=4,
        labels=["low", "mid", "high", "elite"]
    ).astype(str)

bn_cols = [
    c for c in [
        "position", "league", "team", "nationality",
        "is_goalkeeper", "age_bin", "minutes_played_bin",
        "rating_pred_bin"
    ]
    if c in df_bn.columns
]

bn_ready = df_bn[bn_cols].copy()

print("BN-ready shape:", bn_ready.shape)


## 10. Sınıflandırma veri setinin hazırlanması

Hedef değişken olarak `rating_pred_bin` kullanılmıştır.

Target leakage oluşmaması için `rating_pred` doğrudan model girdilerinden çıkarılmıştır.

Orijinal train/test dağılımı:

- Train: **26.569 satır**
- Test: **6.643 satır**


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

df = df_iqr_clean.copy()

df["rating_pred_bin"] = pd.qcut(
    df["rating_pred"],
    q=4,
    labels=["low", "mid", "high", "elite"]
)

TARGET = "rating_pred_bin"

drop_cols = [TARGET, "rating_pred"]
for c in ["id", "name"]:
    if c in df.columns:
        drop_cols.append(c)

X = df.drop(columns=drop_cols)
y = df[TARGET].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ]
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)


## 11. Logistic Regression

**Orijinal test accuracy sonucu: %70,37**


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

clf_logreg = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=2000))
])

clf_logreg.fit(X_train, y_train)
pred_logreg = clf_logreg.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred_logreg))
print(classification_report(y_test, pred_logreg))


## 12. Random Forest

Random Forest modeli şu parametrelerle oluşturulmuştur:

- `n_estimators=300`
- `random_state=42`
- `n_jobs=-1`
- `class_weight="balanced"`

**Orijinal test accuracy sonucu: %85,79**


In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf_rf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

clf_rf.fit(X_train, y_train)
pred_rf = clf_rf.predict(X_test)

print("RF Accuracy:", accuracy_score(y_test, pred_rf))
print(classification_report(y_test, pred_rf))


## 13. Model karşılaştırması ve Feature Importance

Orijinal çalışmada test accuracy değerlerine göre Random Forest, Logistic Regression modelinden daha başarılı sonuç vermiştir.

Korunan sonuçlar:

- Logistic Regression: **0.7037**
- Random Forest: **0.8579**

Random Forest Feature Importance çıktısında en önemli değişkenlerden biri `minutes_played` olmuştur.


In [ ]:
import joblib

acc_logreg = accuracy_score(y_test, pred_logreg)
acc_rf = accuracy_score(y_test, pred_rf)

print("Logistic Regression Accuracy:", round(acc_logreg, 4))
print("Random Forest Accuracy:", round(acc_rf, 4))

joblib.dump(clf_rf, "final_classification_model.pkl")

pre = clf_rf.named_steps["preprocess"]
rf_model = clf_rf.named_steps["model"]

feature_names = pre.get_feature_names_out()
importance = pd.Series(rf_model.feature_importances_, index=feature_names)
top20 = importance.sort_values(ascending=False).head(20)

display(top20.to_frame("importance"))


## 14. 5-Fold Cross Validation

Random Forest için orijinal 5-Fold Cross Validation sonuçları:

- Ortalama accuracy: **0.6396**
- Standart sapma: **0.0352**

Fold skorları:

`0.6779, 0.6088, 0.5941, 0.6807, 0.6369`


In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    clf_rf,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("5-Fold CV mean accuracy:", round(scores.mean(), 4))
print("5-Fold CV std:", round(scores.std(), 4))
print("Scores:", [round(s, 4) for s in scores])


## 15. TAN Bayesian Network

Orijinal çalışmada `pgmpy` kullanılarak Tree-Augmented Naive Bayes (TAN) Bayesian Network modeli oluşturulmuştur.

Korunan sonuçlar:

- Bayesian veri seti: 8 sütun
- Öğrenilen kenar sayısı: 13
- Model doğrulama sonucu: `True`
- Test accuracy: **%24,26**

Bu model, orijinal test bölmesinde Random Forest modelinden daha düşük performans göstermiştir.


In [ ]:
# Optional Bayesian-network section.
# Requires: networkx and pgmpy

import networkx as nx
from pgmpy.estimators import TreeSearch, BayesianEstimator
from pgmpy.models import DiscreteBayesianNetwork

TARGET = "rating_pred_bin"

bn_df = bn_ready.astype("object")

train_df, test_df = train_test_split(
    bn_df,
    test_size=0.20,
    random_state=42,
    stratify=bn_df[TARGET]
)

root_node = next(col for col in train_df.columns if col != TARGET)

est = TreeSearch(train_df, root_node=root_node)
dag = est.estimate(estimator_type="tan", class_node=TARGET)

model = DiscreteBayesianNetwork(dag.edges())
model.fit(
    train_df,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=10
)

print("Learned edge count:", len(dag.edges()))
print("Model valid:", model.check_model())


## Proje Kurtarma Durumu

Bu notebook orijinal projenin yöntemlerini ve kaydedilmiş sonuçlarını korumaktadır.

Ham veri seti daha sonra bulunursa `data/archive.zip` konumuna eklenerek notebook baştan sona yeniden çalıştırılabilir.
